In [ ]:
!pip install catboost xgboost lightgbm
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
import plotly.graph_objects as go
from sklearn.cluster import KMeans
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from collections import Counter
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
import seaborn as sns
from sklearn.model_selection import train_test_split
!pip install xgboost lightgbm catboost
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split , GridSearchCV ,cross_val_score , RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_squared_error , r2_score
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor
sns.set_style('darkgrid')
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')



In [ ]:
#Data Loading
df = pd.read_csv('data.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (4600, 18)

Columns: ['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city', 'statezip', 'country']

First few rows:


,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,sqft_basement,yr_built,yr_renovated,street,city,statezip,country
0,2014-05-02 00:00:00,313000.0,3.0,1.50,1340,7912,1.5,0,0,3,1340,0,1955,2005,18810 Densmore Ave N,Shoreline,WA 98133,USA
1,2014-05-02 00:00:00,2384000.0,5.0,2.50,3650,9050,2.0,0,4,5,3370,280,1921,0,709 W Blaine St,Seattle,WA 98119,USA
2,2014-05-02 00:00:00,342000.0,3.0,2.00,1930,11947,1.0,0,0,4,1930,0,1966,0,26206-26214 143rd Ave SE,Kent,WA 98042,USA
3,2014-05-02 00:00:00,420000.0,3.0,2.25,2000,8030,1.0,0,0,4,1000,1000,1963,0,857 170th Pl NE,Bellevue,WA 98008,USA
4,2014-05-02 00:00:00,550000.0,4.0,2.50,1940,10500,1.0,0,0,4,1140,800,1976,1992,9105 170th Ave NE,Redmond,WA 98052,USA


## === 1. DATA CLEANING & FEATURE ENGINEERING ===


In [ ]:
#Date Transformation
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year


In [ ]:
#Creating Age Features
df['house_age'] = 2025 - df['yr_built']
df['is_renovated'] = (df['yr_renovated'] > 0).astype(int)
df['years_since_renovation'] = np.where(
    df['yr_renovated'] > 0,
    2025 - df['yr_renovated'],
    0
)


In [ ]:
#Advanced Area Features
df['total_sqft'] = df['sqft_above'] + df['sqft_basement']
df['living_to_lot_ratio'] = df['sqft_living'] / df['sqft_lot']
df['basement_ratio'] = df['sqft_basement'] / df['sqft_living']


In [ ]:
#Extracting Zip Code
df['zipcode'] = df['statezip'].str.extract(r'(\d{5})')

In [ ]:
print(f"\n Missing Values:")
print(df.isnull().sum()[df.isnull().sum() > 0])


 Missing Values:
Series([], dtype: int64)


In [ ]:
print(f"\n Price Statistics:")
print(df['price'].describe())


 Price Statistics:
count    4.600000e+03
mean     5.519630e+05
std      5.638347e+05
min      0.000000e+00
25%      3.228750e+05
50%      4.609435e+05
75%      6.549625e+05
max      2.659000e+07
Name: price, dtype: float64


In [ ]:
print(f"\n Zero Values Check:")
print(f"Price = 0: {(df['price'] == 0).sum()}")
print(f"Bedrooms = 0: {(df['bedrooms'] == 0).sum()}")
print(f"Bathrooms = 0: {(df['bathrooms'] == 0).sum()}")
print(f"Sqft_living = 0: {(df['sqft_living'] == 0).sum()}")

#Handling Zero Values in Price
print(f"\n Houses with price = 0:")
zero_price = df[df['price'] == 0]
print(zero_price[['price', 'bedrooms', 'bathrooms', 'sqft_living', 'city']])



 Zero Values Check:
Price = 0: 49
Bedrooms = 0: 2
Bathrooms = 0: 2
Sqft_living = 0: 0

 Houses with price = 0:
      price  bedrooms  bathrooms  sqft_living              city
4354    0.0       3.0       1.75         1490       Federal Way
4356    0.0       4.0       2.75         2600           Seattle
4357    0.0       6.0       2.75         3200            Burien
4358    0.0       5.0       3.50         3480          Issaquah
4361    0.0       5.0       1.50         1500            Burien
4362    0.0       4.0       4.00         3680         Sammamish
4374    0.0       2.0       2.50         2200          Enumclaw
4376    0.0       4.0       2.25         2170     Normandy Park
4382    0.0       5.0       4.50         4630        Snoqualmie
4383    0.0       5.0       4.00         4430          Bellevue
4385    0.0       4.0       4.50         5030     Mercer Island
4386    0.0       4.0       1.50         2180              Kent
4389    0.0       4.0       3.50         4210          B

In [ ]:
# # Drop rows where house price is zero (treated as actual missing/corrupted values)
df = df[df['price'] > 0].copy()
print(f"Dataset after removing zero prices: {df.shape}")


Dataset after removing zero prices: (4551, 27)


In [ ]:
#   Capping
def cap_outliers(df, column, lower_q=0.01, upper_q=0.99):
    """Caps outlier values in a specified dataframe column using percentile thresholds (Winsorization).
    This preserves data size while minimizing the impact of extreme values."""
    lower = df[column].quantile(lower_q)
    upper = df[column].quantile(upper_q)
    df[column] = df[column].clip(lower=lower, upper=upper)
    return df

# # Define the list of continuous numerical columns prone to outliers
numeric_cols = ['price', 'sqft_living', 'sqft_lot', 'sqft_above',
                'sqft_basement', 'bedrooms', 'bathrooms']

for col in numeric_cols:
    df = cap_outliers(df, col)

print(f"\n✅ Data cleaning complete!")
print(f"Final shape: {df.shape}")
print(f"\nPrice statistics after cleaning:")
print(df['price'].describe())



✅ Data cleaning complete!
Final shape: (4551, 27)

Price statistics after cleaning:
count    4.551000e+03
mean     5.440616e+05
std      3.249697e+05
min      1.480000e+05
25%      3.262643e+05
50%      4.650000e+05
75%      6.575000e+05
max      2.016000e+06
Name: price, dtype: float64


# Visualization

In [ ]:
counts = df['house_age'].value_counts().reset_index()
counts.columns = ['house_age', 'count']

fig = px.bar(counts,
             x='house_age',
             y='count',
             title="Distribution of Houses by Number of house_age",
             labels={'house_age': 'Number of house_age', 'count': 'Number of Houses'},
             color='count',
             height=400,
             template="plotly_white")

fig.show()

In [ ]:
fig = px.histogram(df,
                   x="price",
                   title="price distribution",
                   color_discrete_sequence=['indianred'], # لون واحد لكل الأعمدة
                   marginal="box")
fig.show()

In [ ]:
# Create a box plot using Plotly Express
fig = px.box(df, y="price")
fig.show()

In [ ]:

import plotly.express as px


counts = df['bedrooms'].value_counts().reset_index()
counts.columns = ['bedrooms', 'count']

fig = px.bar(counts,
             x='bedrooms',
             y='count',
             title="Distribution of Houses by Number of Bedrooms",
             labels={'bedrooms': 'Number of Bedrooms', 'count': 'Number of Houses'},
             color='count',
             height=400,
             template="plotly_white")

fig.show()

In [ ]:
import plotly.express as px

bath_counts = df['bathrooms'].value_counts().reset_index()
bath_counts.columns = ['bathrooms', 'count']


bath_counts = bath_counts.sort_values('bathrooms')

fig = px.bar(bath_counts,
             x=bath_counts['bathrooms'].astype(str),
             y='count',
             title="Distribution of Houses by Number of Bathrooms",
             labels={'x': 'Number of Bathrooms (with fractions)', 'count': 'Number of Houses'},
             color='count',
             height=500,
             template="plotly_white")

fig.update_xaxes(tickangle=45)

fig.show()

In [ ]:
counts = df['floors'].value_counts().reset_index()
counts.columns = ['floors', 'count']

fig = px.bar(counts,
             x='floors',
             y='count',
             title="Distribution of Houses by Number of floors",
             labels={'floors': 'Number of floors', 'count': 'Number of Houses'},
             color='count',
             height=400,
             template="plotly_white")

fig.show()

In [ ]:
feature ='city'
df_counts = df[feature].value_counts().reset_index()
df_counts.columns = [feature, 'count']


limit = 10
df_counts.loc[df_counts['count'] < limit, feature] = 'Other cities'

fig = px.pie(df_counts,
             values='count',
             names=feature,
             title='Distribution of houses by {feature} (with the merging of small towns)')

fig.show()

In [ ]:
import plotly.express as px

feature1 = 'price'            #like price
feature2 = 'zipcode'
# 1. Grouping by feature2 and calculating the average feature1
feVSfe = df.groupby(feature2)[feature1].mean().reset_index()

# 2. Sorting to show the Top 10 most expensive areas
feVSfe = feVSfe.sort_values(by=feature1, ascending=False).head(10)

# 3. Creating the Pie (Donut) Chart
fig = px.pie(feVSfe,
             values=feature1,
             names=feature2,
             title='Top 10 {feature2} by Average{feature1}',
             hole=0.4,
             color_discrete_sequence=px.colors.sequential.RdBu,
             template="plotly_white")

# 4. Updating labels to show percentage and the feature2
fig.update_traces(textinfo='percent+label')

# 5. Fixing the margins for a cleaner look
fig.update_layout(margin=dict(t=50, b=20, l=20, r=20))

fig.show()
##
fig = px.bar(feVSfe,
             x=feature2,
             y=feature1,
             title=f'Average {feature1.capitalize()} per {feature2.capitalize()} (Top 10)',
             color=feature1,
             labels={feature1: f'Average {feature1.capitalize()}', feature2: feature2.capitalize()},
             template="plotly_white",
             height=500)

fig.show()

In [ ]:
# 3. (Top 15 Cities)
avg_city_price = df.groupby('city')['price'].mean().sort_values(ascending=False).head(15).reset_index()

fig_city = px.bar(avg_city_price,
                 x='city',
                 y='price',
                 color='price',
                 text_auto='.3s',
                 title="<b>Strategic Insight:</b> Top 15 High-Value Cities for Investment",
                 labels={"city": "City", "price": "Average Price ($)"},
                 template="plotly_dark",
                 color_continuous_scale='Blues')


fig_city.update_layout(
    title_x=0.5,
    title_font_size=22,
    xaxis_tickangle=-45,
    font=dict(family="Arial", size=13),
    coloraxis_showscale=False
)


fig_city.update_traces(marker_line_color='rgb(8,48,107)',
                  marker_line_width=1.5, opacity=0.9)

fig_city.show()

In [ ]:

# 4. (Waterfront Impact)
fig_water = px.box(df,
                  x="waterfront",
                  y="price",
                  color="waterfront",
                  notched=True,
                  title="<b>Feature Analysis:</b> Does a Waterfront View Justify a Premium Price?",
                  labels={"waterfront": "Waterfront View (0=No, 1=Yes)", "price": "Price ($)"},
                  template="plotly_dark",
                  color_discrete_sequence=['#63ace5', '#084594'])


fig_water.update_layout(
    title_x=0.5,
    title_font_size=22,
    showlegend=False,
    font=dict(family="Arial", size=14),
    yaxis=dict(gridcolor='rgba(255,255,255,0.1)')
)


fig_water.update_traces(marker_size=4, line_width=2)

fig_water.show()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4551 entries, 0 to 4599
Data columns (total 27 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   date                    4551 non-null   datetime64[ns]
 1   price                   4551 non-null   float64       
 2   bedrooms                4551 non-null   float64       
 3   bathrooms               4551 non-null   float64       
 4   sqft_living             4551 non-null   int64         
 5   sqft_lot                4551 non-null   float64       
 6   floors                  4551 non-null   float64       
 7   waterfront              4551 non-null   int64         
 8   view                    4551 non-null   int64         
 9   condition               4551 non-null   int64         
 10  sqft_above              4551 non-null   int64         
 11  sqft_basement           4551 non-null   int64         
 12  yr_built                4551 non-null   int64        

In [ ]:
df.drop(columns=['date', 'statezip','year','country','street','yr_renovated'], inplace=True)
df.head()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,sqft_above,...,yr_built,city,month,house_age,is_renovated,years_since_renovation,total_sqft,living_to_lot_ratio,basement_ratio,zipcode
0,313000.0,3.0,1.50,1340,7912.0,1.5,0,0,3,1340,...,1955,Shoreline,5,70,1,20,1340,0.169363,0.000000,98133
1,2016000.0,5.0,2.50,3650,9050.0,2.0,0,4,5,3370,...,1921,Seattle,5,104,0,0,3650,0.403315,0.076712,98119
2,342000.0,3.0,2.00,1930,11947.0,1.0,0,0,4,1930,...,1966,Kent,5,59,0,0,1930,0.161547,0.000000,98042
3,420000.0,3.0,2.25,2000,8030.0,1.0,0,0,4,1000,...,1963,Bellevue,5,62,0,0,2000,0.249066,0.500000,98008
4,550000.0,4.0,2.50,1940,10500.0,1.0,0,0,4,1140,...,1976,Redmond,5,49,1,33,1940,0.184762,0.412371,98052


In [ ]:

# 5.(Correlation Heatmap)
corr = df.select_dtypes(include=[np.number]).corr()

fig_corr = go.Figure(data=go.Heatmap(
                   z=corr.values,
                   x=corr.columns,
                   y=corr.columns,
                   colorscale='RdBu',
                   zmin=-1, zmax=1,
                   text=np.round(corr.values, 2),
                   texttemplate="%{text}",
                   hoverongaps=False))


fig_corr.update_layout(
    title="<b>Market Map:</b> Correlation Between Features and Price",
    title_x=0.5,
    title_font_size=22,
    template="plotly_dark",
    height=800,
    xaxis_tickangle=-45,
    font=dict(family="Arial", size=12)
)


fig_corr.update_traces(colorbar_title="Correlation Score", colorbar_thickness=20)

fig_corr.show()

In [ ]:
import pandas as pd

# 1. Define the IQR function (same as yours but cleaned)
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Filter outliers
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers

# 2. List of numeric columns from your data
# We excluded strings like 'street', 'city', 'statezip'
numeric_cols = [
    'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
    'floors', 'waterfront', 'view', 'condition', 'sqft_above',
    'sqft_basement', 'house_age',
]

# 3. Loop through all columns and print the number of outliers
print("--- Outlier Detection Report ---")
for col in numeric_cols:
    outliers_df = detect_outliers_iqr(df, col)
    count = outliers_df.shape[0]
    percentage = (count / df.shape[0]) * 100
    print(f"{col:15} | Outliers: {count:4} | Percentage: {percentage:.2f}%")

--- Outlier Detection Report ---
price           | Outliers:  240 | Percentage: 5.27%
bedrooms        | Outliers:   76 | Percentage: 1.67%
bathrooms       | Outliers:  128 | Percentage: 2.81%
sqft_living     | Outliers:  128 | Percentage: 2.81%
sqft_lot        | Outliers:  540 | Percentage: 11.87%
floors          | Outliers:    0 | Percentage: 0.00%
waterfront      | Outliers:   30 | Percentage: 0.66%
view            | Outliers:  448 | Percentage: 9.84%
condition       | Outliers:    6 | Percentage: 0.13%
sqft_above      | Outliers:  111 | Percentage: 2.44%
sqft_basement   | Outliers:   83 | Percentage: 1.82%
house_age       | Outliers:    0 | Percentage: 0.00%


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4551 entries, 0 to 4599
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   price                   4551 non-null   float64
 1   bedrooms                4551 non-null   float64
 2   bathrooms               4551 non-null   float64
 3   sqft_living             4551 non-null   int64  
 4   sqft_lot                4551 non-null   float64
 5   floors                  4551 non-null   float64
 6   waterfront              4551 non-null   int64  
 7   view                    4551 non-null   int64  
 8   condition               4551 non-null   int64  
 9   sqft_above              4551 non-null   int64  
 10  sqft_basement           4551 non-null   int64  
 11  yr_built                4551 non-null   int64  
 12  city                    4551 non-null   object 
 13  month                   4551 non-null   int32  
 14  house_age               4551 non-null   int64

## === Clustering ===

In [ ]:
df['is_renovated'].value_counts()

,count
is_renovated,
0,2706
1,1845


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import silhouette_score

#  3. city → Target Encoding
city_mean_price = df.groupby('city')['price'].mean()
df['city_encoded'] = df['city'].map(city_mean_price)


le = LabelEncoder()
df['city_encoded'] = le.fit_transform(df['city'])

binary_cols = ['waterfront', 'is_renovated']  # 0,1

scale_cols = [
    'sqft_living', 'sqft_lot', 'bedrooms', 'bathrooms',
    'floors', 'view', 'condition', 'house_age',
    'city_encoded',
    'total_sqft', 'living_to_lot_ratio', 'basement_ratio'
]

X_cluster = df[scale_cols + binary_cols].copy()
X_cluster = X_cluster.fillna(X_cluster.median())

scaler = StandardScaler()
X_scaled_continuous = scaler.fit_transform(X_cluster[scale_cols])

X_scaled = np.hstack([
    X_scaled_continuous,
    X_cluster[binary_cols].values
])

#  K
inertias = []
silhouettes = []
K_range = range(2, 10)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, kmeans.labels_))

# Visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Elbow Method', 'Silhouette Score'),
    specs=[[{"type": "scatter"}, {"type": "scatter"}]]
)

fig.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
                        name='Inertia', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=list(K_range), y=silhouettes, mode='lines+markers',
                        name='Silhouette', line=dict(color='red')), row=1, col=2)

fig.update_layout(height=400, title_text="Optimal K Selection", showlegend=False)
fig.show()

print("\n📊 Silhouette Scores:")
for k, score in zip(K_range, silhouettes):
    print(f"K={k}: {score:.3f}")




📊 Silhouette Scores:
K=2: 0.216
K=3: 0.204
K=4: 0.207
K=5: 0.218
K=6: 0.232
K=7: 0.205
K=8: 0.197
K=9: 0.189


In [ ]:
from sklearn.decomposition import PCA

optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

cluster_names = {
    2: "Budget Old City",       # Small, old, affordable
    3: "Mid-range Suburban",    # Mid-sized, slightly older
    0: "Upper Mid-range New",   # Large, new, moderate price
    1: "Luxury Spacious"        # Largest, most expensive, premium view
}

df['cluster_name'] = df['cluster'].map(cluster_names)

cluster_counts = df['cluster'].value_counts().sort_index()
print(" Cluster Distribution (K=4):")
for k in range(optimal_k):
    count = cluster_counts[k]
    print(f"  Cluster {k} - {cluster_names[k]}: {count} ({count/len(df)*100:.1f}%)")

# PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]


fig = px.scatter(
    df, x='PC1', y='PC2',
    color='cluster_name',
    title='Property Clusters - K=4 (PCA)',
    labels={
        'PC1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
        'PC2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'
    },
    color_discrete_sequence=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'],
    opacity=0.6,
    width=900, height=600
)

fig.update_traces(marker=dict(size=5, line=dict(width=0.5, color='white')))
fig.update_layout(legend_title_text='Cluster')
fig.show()

print(f"\nPCA Explained Variance: {pca.explained_variance_ratio_.cumsum()[1]*100:.1f}%")


 Cluster Distribution (K=4):
  Cluster 0 - Upper Mid-range New: 1488 (32.7%)
  Cluster 1 - Luxury Spacious: 488 (10.7%)
  Cluster 2 - Budget Old City: 1428 (31.4%)
  Cluster 3 - Mid-range Suburban: 1147 (25.2%)



PCA Explained Variance: 48.1%


In [ ]:
# Cluster Profiles
cluster_profiles = df.groupby('cluster').agg({
    'price': ['mean', 'median', 'min', 'max', 'count'],
    'sqft_living': 'mean',
    'bedrooms': 'mean',
    'bathrooms': 'mean',
    'floors': 'mean',
    'waterfront': 'mean',
    'view': 'mean',
    'condition': 'mean',
    'house_age': 'mean',
    'is_renovated': 'mean'
}).round(2)

cluster_profiles.columns = ['_'.join(col).strip() for col in cluster_profiles.columns]
cluster_profiles = cluster_profiles.reset_index()
cluster_profiles['cluster_name'] = cluster_profiles['cluster'].map(cluster_names)
print(cluster_profiles.to_string(index=False))


 cluster  price_mean  price_median  price_min  price_max  price_count  sqft_living_mean  bedrooms_mean  bathrooms_mean  floors_mean  waterfront_mean  view_mean  condition_mean  house_age_mean  is_renovated_mean        cluster_name
       0   545018.63      500000.0   148000.0  2016000.0         1488           2284.96           3.47            2.53         2.07             0.00       0.04            3.12           27.60               0.23 Upper Mid-range New
       1  1035927.02      920000.0   148000.0  2016000.0          488           3756.67           4.22            3.20         1.80             0.04       1.31            3.38           44.87               0.37     Luxury Spacious
       2   376253.68      340000.0   148000.0  2016000.0         1428           1332.23           2.80            1.41         1.14             0.00       0.09            3.52           72.59               0.56     Budget Old City
       3   542470.61      495000.0   161700.0  2016000.0         1147       

In [ ]:
#  1. Price Distribution
fig1 = px.histogram(df, x='price', color='cluster_name',
                    barmode='overlay', nbins=30,
                    title='Price Distribution by Cluster',
                    range_x=[0, 2000000], opacity=0.6)
fig1.show()

In [ ]:
#  2. Sqft Living Box Plot
fig2 = px.box(df, x='cluster_name', y='sqft_living',
              color='cluster_name',
              title='Living Area by Cluster')
fig2.show()

In [ ]:
#  3. Bedrooms Distribution
fig3 = px.histogram(df, x='bedrooms', color='cluster_name',
                    barmode='group',
                    title='Bedrooms Distribution by Cluster')
fig3.show()


In [ ]:
#  4. Waterfront & View
waterfront_view = df.groupby('cluster_name')[['waterfront', 'view']].mean().reset_index()
fig4 = px.bar(waterfront_view.melt(id_vars='cluster_name'),
              x='cluster_name', y='value', color='variable',
              barmode='group',
              title='Waterfront & View Score by Cluster')
fig4.show()

In [ ]:
# === 4.1 تحضير البيانات ===
X = pd.DataFrame(X_scaled, columns=scale_cols + binary_cols)
y = df['cluster']

print("📊 Original Class Distribution:")
print(Counter(y))
print(f"Imbalance Ratio: {Counter(y)[1] / Counter(y)[3]:.1f}:1 (Majority:Minority)")

# === 4.2 Stratified Split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Train set: {Counter(y_train)}")
print(f"📊 Test set: {Counter(y_test)}")

📊 Original Class Distribution:
Counter({0: 1488, 2: 1428, 3: 1147, 1: 488})
Imbalance Ratio: 0.4:1 (Majority:Minority)

📊 Train set: Counter({0: 1190, 2: 1142, 3: 918, 1: 390})
📊 Test set: Counter({0: 298, 2: 286, 3: 229, 1: 98})


In [ ]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Random Forest (Regularized)': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,              # تقييد العمق
        min_samples_split=10,      # زيادة العينات للتقسيم
        min_samples_leaf=4,        # زيادة الأوراق
        max_features='sqrt',       # تقييد الميزات
        class_weight='balanced',   # Class weights
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting (Regularized)': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,        # معدل تعلم بطيء
        max_depth=5,               # تقييد العمق
        min_samples_split=10,
        subsample=0.8,             # Stochastic
        random_state=42
    ),
    'Logistic Regression (Regularized)': LogisticRegression(
        max_iter=2000,
        C=0.1,                     # Regularization قوي
        class_weight='balanced',
        random_state=42
    )
}

results = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"🔧 Training: {name}")
    print(f"{'='*50}")

    cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='balanced_accuracy')

    # ✅ X_train و y_train بدل X_train_bal و y_train_bal
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_bal_acc = balanced_accuracy_score(y_train, y_pred_train)
    test_balanced_acc = balanced_accuracy_score(y_test, y_pred_test)

    overfitting_gap = train_bal_acc - test_balanced_acc

    results[name] = {
        'CV Balanced Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Train Acc (Balanced)': train_bal_acc,
        'Test Balanced Acc': test_balanced_acc,
        'True Gap': overfitting_gap
    }

    print(f"  CV Balanced Accuracy: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})")
    print(f"  Train (Balanced): {train_bal_acc:.3f}")
    print(f"  Test (Balanced): {test_balanced_acc:.3f}")
    print(f"  True Overfitting Gap: {overfitting_gap:.3f} {'✅' if overfitting_gap < 0.05 else '⚠️'}")

comparison = pd.DataFrame(results).T
print(comparison.round(3).to_string())



🔧 Training: Random Forest (Regularized)
  CV Balanced Accuracy: 0.940 (+/- 0.037)
  Train (Balanced): 0.981
  Test (Balanced): 0.953
  True Overfitting Gap: 0.029 ✅

🔧 Training: Gradient Boosting (Regularized)
  CV Balanced Accuracy: 0.949 (+/- 0.031)
  Train (Balanced): 1.000
  Test (Balanced): 0.959
  True Overfitting Gap: 0.041 ✅

🔧 Training: Logistic Regression (Regularized)
  CV Balanced Accuracy: 0.981 (+/- 0.010)
  Train (Balanced): 0.987
  Test (Balanced): 0.989
  True Overfitting Gap: -0.002 ✅
                                   CV Balanced Mean  CV Std  Train Acc (Balanced)  Test Balanced Acc  True Gap
Random Forest (Regularized)                   0.940   0.018                 0.981              0.953     0.029
Gradient Boosting (Regularized)               0.949   0.016                 1.000              0.959     0.041
Logistic Regression (Regularized)             0.981   0.005                 0.987              0.989    -0.002


In [ ]:
best_model_name = 'Logistic Regression (Regularized)'
best_classifier = models[best_model_name]

print(f"🏆 BEST MODEL: {best_model_name}")
print(f"   Train Balanced Accuracy: {results[best_model_name]['Train Acc (Balanced)']:.3f}")
print(f"   Test Balanced Accuracy: {results[best_model_name]['Test Balanced Acc']:.3f}")


🏆 BEST MODEL: Logistic Regression (Regularized)
   Train Balanced Accuracy: 0.987
   Test Balanced Accuracy: 0.989


In [ ]:
# Predictions
y_pred = best_classifier.predict(X_test)


In [ ]:
# Classification Report
print(f"\n{'='*60}")
print("📋 CLASSIFICATION REPORT")
print(f"{'='*60}")
target_names = [cluster_names[i] for i in sorted(cluster_names.keys())]
print(classification_report(y_test, y_pred, target_names=target_names, digits=3))



📋 CLASSIFICATION REPORT
                     precision    recall  f1-score   support

Upper Mid-range New      0.997     0.977     0.986       298
    Luxury Spacious      0.970     1.000     0.985        98
    Budget Old City      0.990     0.990     0.990       286
 Mid-range Suburban      0.978     0.991     0.985       229

           accuracy                          0.987       911
          macro avg      0.984     0.989     0.986       911
       weighted avg      0.987     0.987     0.987       911



In [ ]:
# Confusion Matrix
fig_cm = px.imshow(cm,
    labels=dict(x="Predicted", y="Actual"),
    x=target_names, y=target_names,
    text_auto=True,
    title=f'Confusion Matrix - {best_model_name}',
    color_continuous_scale='Blues')
fig_cm.show()

In [ ]:
feature_importance = pd.DataFrame({
    'feature': scale_cols + binary_cols,
    'importance': np.abs(best_classifier.coef_).mean(axis=0)  #
}).sort_values('importance', ascending=False)

In [ ]:
# Feature Importance
fig_fi = px.bar(feature_importance.head(10),
    x='importance', y='feature',
    orientation='h',
    title='Top 10 Features for Cluster Classification',
    color='importance',
    color_continuous_scale='viridis')
fig_fi.show()

print(f"\n🔝 Top 5 Important Features:")
print(feature_importance.head(5).to_string(index=False))


🔝 Top 5 Important Features:
       feature  importance
basement_ratio    1.102273
        floors    0.949703
   sqft_living    0.893919
     bathrooms    0.870155
    total_sqft    0.836878


In [ ]:
joblib.dump(best_classifier, 'classifier_model.pkl')
joblib.dump(scaler, 'cluster_scaler.pkl')
joblib.dump(kmeans, 'kmeans_model.pkl')

print("💾 Models saved:")
print("   ✅ classifier_model.pkl")
print("   ✅ cluster_scaler.pkl")
print("   ✅ kmeans_model.pkl")

In [ ]:


# === 5.1 PRICE REGRESSION PREPARATION ===
print(f"\n{'='*60}")
print("💰 STARTING PRICE REGRESSION")
print(f"{'='*60}")

# 1. Transform categorical cluster labels via One-Hot Encoding
# 'drop_first=True' is utilized to strictly prevent the multicollinearity (Dummy Variable Trap)
df_reg_features = df[scale_cols + binary_cols + ['cluster']].copy()
df_reg_features = pd.get_dummies(df_reg_features, columns=['cluster'], drop_first=True)

# Register the expanded feature list after dynamic dummy generation
regression_features = df_reg_features.columns.tolist()

X_reg = df_reg_features
# Apply log1p transformation to mitigate severe target skewness in house prices
y_reg = np.log1p(df['price'])

# 2. Segment into features and target training/testing splits chronologically
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# 3. Secure Imputation against Data Leakage
# Compute median metrics strictly from the training partition to fill potential missing structures
train_medians = X_train_r.median()
X_train_r = X_train_r.fillna(train_medians)
X_test_r = X_test_r.fillna(train_medians)

# Log descriptive structural data and operational targets
print(f"Train size: {len(X_train_r)}, Test size: {len(X_test_r)}")
print(f"Price range: ${df['price'].min():,.0f} - ${df['price'].max():,.0f}")
print(f"Log Price range: {y_reg.min():.3f} - {y_reg.max():.3f}")


💰 STARTING PRICE REGRESSION
Train size: 3640, Test size: 911
Price range: $148,000 - $2,016,000
Log Price range: 11.905 - 14.517


In [ ]:
regression_features = [
    'sqft_living', 'sqft_lot', 'bedrooms', 'bathrooms',
    'floors', 'waterfront', 'view', 'condition',
    'house_age', 'is_renovated', 'years_since_renovation',
    'total_sqft', 'living_to_lot_ratio', 'basement_ratio',
    'city_encoded', 'cluster'
]

X_reg = df[regression_features].copy()
y_reg = np.log1p(df['price'])  # log transform

In [ ]:
# === 5.2 Train/Test Split ===
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Imputation = train
train_medians = X_train_r.median()
X_train_r = X_train_r.fillna(train_medians)
X_test_r = X_test_r.fillna(train_medians)

print(f"Train size: {len(X_train_r)}, Test size: {len(X_test_r)}")
print(f"Price range: ${df['price'].min():,.0f} - ${df['price'].max():,.0f}")

Train size: 3640, Test size: 911
Price range: $148,000 - $2,016,000


In [ ]:
def evaluate_reg(y_true, y_pred):
    y_true_orig = np.expm1(y_true)
    y_pred_orig = np.expm1(y_pred)
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true_orig, y_pred_orig)),
        'MAE':  mean_absolute_error(y_true_orig, y_pred_orig),
        'MAPE': np.mean(np.abs((y_true_orig - y_pred_orig) / y_true_orig)) * 100,
        'R2':   r2_score(y_true, y_pred)
    }


In [ ]:
regressors = {
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=12,
        min_samples_split=10, min_samples_leaf=4,
        max_features='sqrt', random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05,
        max_depth=6, min_samples_split=10,
        subsample=0.8, random_state=42
    ),
    'Ridge Regression': Ridge(alpha=1.0),
    'CatBoost': CatBoostRegressor(
        iterations=1000, learning_rate=0.05,
        depth=6, l2_leaf_reg=10,
        random_seed=42, verbose=0,
        early_stopping_rounds=50
    ),
    'XGBoost': XGBRegressor(
        n_estimators=1000, learning_rate=0.05,
        max_depth=6, reg_lambda=10,
        subsample=0.8, random_state=42,
        early_stopping_rounds=50,
        verbosity=0
    ),
    'LightGBM': LGBMRegressor(
        n_estimators=1000, learning_rate=0.05,
        max_depth=6, reg_lambda=10,
        subsample=0.8, random_state=42,
        verbose=-1
    )
}

reg_results = {}
for name, model in regressors.items():
    print(f"\n{'='*50}")
    print(f"🔧 Training: {name}")
    print(f"{'='*50}")


    if name in ['CatBoost', 'XGBoost']:
        model.fit(X_train_r, y_train_r, eval_set=[(X_test_r, y_test_r)])
    else:
        model.fit(X_train_r, y_train_r)

    train_m = evaluate_reg(y_train_r, model.predict(X_train_r))
    test_m  = evaluate_reg(y_test_r,  model.predict(X_test_r))
    gap = (test_m['RMSE'] - train_m['RMSE']) / test_m['RMSE']

    reg_results[name] = {
        'Train RMSE ($)': train_m['RMSE'],
        'Test RMSE ($)':  test_m['RMSE'],
        'Test MAE ($)':   test_m['MAE'],
        'Test MAPE (%)':  test_m['MAPE'],
        'Test R2':        test_m['R2'],
        'Gap':            gap
    }

    print(f"  Train RMSE: ${train_m['RMSE']:,.0f}")
    print(f"  Test RMSE:  ${test_m['RMSE']:,.0f}")
    print(f"  Test MAE:   ${test_m['MAE']:,.0f}")
    print(f"  Test MAPE:  {test_m['MAPE']:.2f}%")
    print(f"  Test R²:    {test_m['R2']:.3f}")
    print(f"  Gap:        {gap:.1%} {'✅' if gap < 0.15 else '⚠️'}")



🔧 Training: Random Forest
  Train RMSE: $158,915
  Test RMSE:  $212,151
  Test MAE:   $134,474
  Test MAPE:  25.46%
  Test R²:    0.616
  Gap:        25.1% ⚠️

🔧 Training: Gradient Boosting
  Train RMSE: $87,651
  Test RMSE:  $183,680
  Test MAE:   $105,164
  Test MAPE:  19.75%
  Test R²:    0.712
  Gap:        52.3% ⚠️

🔧 Training: Ridge Regression
  Train RMSE: $216,050
  Test RMSE:  $219,963
  Test MAE:   $146,674
  Test MAPE:  28.78%
  Test R²:    0.552
  Gap:        1.8% ✅

🔧 Training: CatBoost
  Train RMSE: $129,683
  Test RMSE:  $181,879
  Test MAE:   $103,876
  Test MAPE:  19.68%
  Test R²:    0.720
  Gap:        28.7% ⚠️

🔧 Training: XGBoost
[0]	validation_0-rmse:0.51258
[1]	validation_0-rmse:0.50053
[2]	validation_0-rmse:0.48890
[3]	validation_0-rmse:0.47825
[4]	validation_0-rmse:0.46825
[5]	validation_0-rmse:0.45858
[6]	validation_0-rmse:0.44891
[7]	validation_0-rmse:0.44013
[8]	validation_0-rmse:0.43260
[9]	validation_0-rmse:0.42487
[10]	validation_0-rmse:0.41813
[11]	vali

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RandomizedSearchCV

# === 1. One-Hot Encoding على city ===
X_reg = df[regression_features].copy()

# بدل city_encoded استخدم OHE
X_reg = pd.get_dummies(X_reg, columns=['city_encoded'], drop_first=True)
# أو الأسهل: شيل city_encoded وحط statezip كـ OHE

y_reg = np.log1p(df['price'])

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

In [ ]:
# === 2. Permutation Importance - شيل الفيتشرز الضعيفة ===
cat_temp = CatBoostRegressor(iterations=300, random_seed=42, verbose=0)
cat_temp.fit(X_train_r, y_train_r)

perm = permutation_importance(
    cat_temp, X_train_r, y_train_r,
    n_repeats=10, random_state=42,
    scoring='neg_mean_squared_error'
)

perm_importance = pd.Series(perm.importances_mean, index=X_train_r.columns)
selector = perm_importance[perm_importance > 0.00].index

X_train_r = X_train_r[selector]
X_test_r  = X_test_r[selector]
print(f"Features بعد الفلترة: {len(selector)}")

Features بعد الفلترة: 53


In [ ]:
# === 3. RandomizedSearchCV على CatBoost ===
param_grid = {
    'iterations':    [500, 800, 1000, 2000, 3000, 4000],
    'learning_rate': [0.03, 0.05, 0.1],
    'depth':         [4, 6, 8, 9],
    'l2_leaf_reg':   [5, 10, 15, 20],
    'subsample':     [0.7, 0.8, 1.0]
}

random_search = RandomizedSearchCV(
    CatBoostRegressor(random_seed=42, verbose=0),
    param_distributions=param_grid,
    n_iter=20, cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1, random_state=42
)

random_search.fit(X_train_r, y_train_r)

print(f"Best Params: {random_search.best_params_}")
best_cat = random_search.best_estimator_

Best Params: {'subsample': 0.7, 'learning_rate': 0.05, 'l2_leaf_reg': 15, 'iterations': 1000, 'depth': 6}


In [ ]:
# === Evaluation ===
y_pred_test = best_cat.predict(X_test_r)
test_m = evaluate_reg(y_test_r, y_pred_test)

print(f"Test RMSE: ${test_m['RMSE']:,.0f}")
print(f"Test MAPE: {test_m['MAPE']:.2f}%")
print(f"Test R²:   {test_m['R2']:.3f}")

Test RMSE: $177,592
Test MAPE: 19.04%
Test R²:   0.730


In [ ]:
"""
====================================================
  SAVE ALL ARTIFACTS FOR DEPLOYMENT
  Run this AFTER your full training pipeline
  It saves everything needed for the Streamlit app
====================================================
"""

import joblib
import json
import numpy as np
import pandas as pd

# ============================================================
# ✅ 1. ALREADY SAVED IN YOUR CODE (just making sure)
# ============================================================
joblib.dump(best_classifier, 'classifier_model.pkl')
joblib.dump(scaler, 'cluster_scaler.pkl')
joblib.dump(kmeans, 'kmeans_model.pkl')
joblib.dump(best_cat, 'regression_model.pkl')
# These are already in your code ✅

# ============================================================
# ✅ 2. ADD THESE LINES TO YOUR CODE AFTER TRAINING
# ============================================================

# --- After the clustering section ---
# Save LabelEncoder for city
joblib.dump(le, 'label_encoder_city.pkl')
print("✅ label_encoder_city.pkl saved")

# Save city mean price mapping (Target Encoding)
city_mean_price_dict = city_mean_price.to_dict()
with open('city_mean_price.json', 'w') as f:
    json.dump(city_mean_price_dict, f)
print("✅ city_mean_price.json saved")

# --- After regression section ---
# Save train medians for imputation
joblib.dump(train_medians, 'train_medians.pkl')
print("✅ train_medians.pkl saved")

# Save selected features list (after permutation importance)
selected_features = selector.tolist()
with open('selected_features.json', 'w') as f:
    json.dump(selected_features, f)
print("✅ selected_features.json saved")

# ============================================================
# ✅ 3. SAVE METADATA (columns, cluster names, etc.)
# ============================================================

metadata = {
    "cluster_names": {
        "0": "Upper Mid-range New",
        "1": "Luxury Spacious",
        "2": "Budget Old City",
        "3": "Mid-range Suburban"
    },
    "scale_cols": [
        'sqft_living', 'sqft_lot', 'bedrooms', 'bathrooms',
        'floors', 'view', 'condition', 'house_age',
        'city_encoded',
        'total_sqft', 'living_to_lot_ratio', 'basement_ratio'
    ],
    "binary_cols": ['waterfront', 'is_renovated'],
    "regression_features": [
        'sqft_living', 'sqft_lot', 'bedrooms', 'bathrooms',
        'floors', 'waterfront', 'view', 'condition',
        'house_age', 'is_renovated', 'years_since_renovation',
        'total_sqft', 'living_to_lot_ratio', 'basement_ratio',
        'city_encoded', 'cluster'
    ]
}

with open('metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("✅ metadata.json saved")

# ============================================================
# ✅ 4. VERIFICATION - Load everything back and test
# ============================================================
print("\n" + "="*50)
print("🔍 VERIFICATION: Loading all artifacts...")
print("="*50)

artifacts = {
    "classifier_model":    joblib.load('classifier_model.pkl'),
    "cluster_scaler":      joblib.load('cluster_scaler.pkl'),
    "kmeans_model":        joblib.load('kmeans_model.pkl'),
    "regression_model":    joblib.load('regression_model.pkl'),
    "label_encoder_city":  joblib.load('label_encoder_city.pkl'),
    "train_medians":       joblib.load('train_medians.pkl'),
}

with open('city_mean_price.json') as f:
    city_mean_price_loaded = json.load(f)

with open('selected_features.json') as f:
    selected_features_loaded = json.load(f)

with open('metadata.json') as f:
    metadata_loaded = json.load(f)

print("✅ classifier_model    →", type(artifacts['classifier_model']).__name__)
print("✅ cluster_scaler      →", type(artifacts['cluster_scaler']).__name__)
print("✅ kmeans_model        →", type(artifacts['kmeans_model']).__name__)
print("✅ regression_model    →", type(artifacts['regression_model']).__name__)
print("✅ label_encoder_city  →", type(artifacts['label_encoder_city']).__name__)
print("✅ train_medians       →", f"{len(artifacts['train_medians'])} features")
print("✅ city_mean_price     →", f"{len(city_mean_price_loaded)} cities")
print("✅ selected_features   →", f"{len(selected_features_loaded)} features")
print("✅ metadata            →", list(metadata_loaded.keys()))

print("\n🎉 ALL ARTIFACTS SAVED AND VERIFIED SUCCESSFULLY!")
print("\n📦 Files to include in your Streamlit app folder:")
files = [
    "classifier_model.pkl",
    "cluster_scaler.pkl",
    "kmeans_model.pkl",
    "regression_model.pkl",
    "label_encoder_city.pkl",
    "train_medians.pkl",
    "city_mean_price.json",
    "selected_features.json",
    "metadata.json",
]
for f in files:
    print(f"   📄 {f}")

✅ label_encoder_city.pkl saved
✅ city_mean_price.json saved
✅ train_medians.pkl saved
✅ selected_features.json saved
✅ metadata.json saved

🔍 VERIFICATION: Loading all artifacts...
✅ classifier_model    → LogisticRegression
✅ cluster_scaler      → StandardScaler
✅ kmeans_model        → KMeans
✅ regression_model    → CatBoostRegressor
✅ label_encoder_city  → LabelEncoder
✅ train_medians       → 16 features
✅ city_mean_price     → 44 cities
✅ selected_features   → 53 features
✅ metadata            → ['cluster_names', 'scale_cols', 'binary_cols', 'regression_features']

🎉 ALL ARTIFACTS SAVED AND VERIFIED SUCCESSFULLY!

📦 Files to include in your Streamlit app folder:
   📄 classifier_model.pkl
   📄 cluster_scaler.pkl
   📄 kmeans_model.pkl
   📄 regression_model.pkl
   📄 label_encoder_city.pkl
   📄 train_medians.pkl
   📄 city_mean_price.json
   📄 selected_features.json
   📄 metadata.json


In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import joblib
import json
import numpy as np
import pandas as pd
import httpx
import os
from typing import Optional

# ─────────────────────────────────────────────
#  App Setup
# ─────────────────────────────────────────────
app = FastAPI(title="Real Estate API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],          # في الـ production خليها domain بتاعك
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ─────────────────────────────────────────────
#  Load Artifacts
# ─────────────────────────────────────────────
classifier   = joblib.load("classifier_model.pkl")
scaler       = joblib.load("cluster_scaler.pkl")
kmeans       = joblib.load("kmeans_model.pkl")
reg_model    = joblib.load("regression_model.pkl")
le_city      = joblib.load("label_encoder_city.pkl")
train_medians = joblib.load("train_medians.pkl")

with open("city_mean_price.json")   as f: city_mean_price   = json.load(f)
with open("selected_features.json") as f: selected_features = json.load(f)
with open("metadata.json")          as f: metadata          = json.load(f)

cluster_names     = metadata["cluster_names"]
scale_cols        = metadata["scale_cols"]
binary_cols       = metadata["binary_cols"]
regression_features = metadata["regression_features"]

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")   # حطها في .env

# ─────────────────────────────────────────────
#  Unit Conversion
# ─────────────────────────────────────────────
SQM_TO_SQFT = 10.7639   # 1 m² = 10.7639 sqft

def to_sqft(m2: float) -> float:
    """Convert square meters to square feet."""
    return m2 * SQM_TO_SQFT

# ─────────────────────────────────────────────
#  Request / Response Schemas
# ─────────────────────────────────────────────
class HouseInput(BaseModel):
    # All area fields are in m² (square meters) — converted to sqft internally
    living_area_m2:   float          # sqft_living
    lot_area_m2:      float          # sqft_lot
    above_area_m2:    float          # sqft_above
    basement_area_m2: float = 0.0   # sqft_basement

    bedrooms:      int
    bathrooms:     float
    floors:        float
    waterfront:    int               # 0 or 1
    view:          int               # 0–4
    condition:     int               # 1–5
    yr_built:      int
    yr_renovated:  int = 0
    city:          str

class PredictResponse(BaseModel):
    cluster_id:      int
    cluster_name:    str
    predicted_price: float
    price_formatted: str
    interpretation:  str
    features_used:   dict   # areas shown in m² for display

# ─────────────────────────────────────────────
#  Feature Engineering Helper
# ─────────────────────────────────────────────
def engineer_features(h: HouseInput) -> dict:
    # ── Convert m² → sqft (model was trained on sqft) ──
    sqft_living   = to_sqft(h.living_area_m2)
    sqft_lot      = to_sqft(h.lot_area_m2)
    sqft_above    = to_sqft(h.above_area_m2)
    sqft_basement = to_sqft(h.basement_area_m2)

    house_age            = 2025 - h.yr_built
    is_renovated         = int(h.yr_renovated > 0)
    years_since_reno     = (2025 - h.yr_renovated) if h.yr_renovated > 0 else 0
    total_sqft           = sqft_above + sqft_basement
    living_to_lot_ratio  = sqft_living / sqft_lot if sqft_lot > 0 else 0
    basement_ratio       = sqft_basement / sqft_living if sqft_living > 0 else 0

    # City encoding
    city_encoded = le_city.transform([h.city])[0] if h.city in le_city.classes_ else 0

    return {
        "sqft_living":          sqft_living,
        "sqft_lot":             sqft_lot,
        "bedrooms":             h.bedrooms,
        "bathrooms":            h.bathrooms,
        "floors":               h.floors,
        "waterfront":           h.waterfront,
        "view":                 h.view,
        "condition":            h.condition,
        "house_age":            house_age,
        "is_renovated":         is_renovated,
        "years_since_renovation": years_since_reno,
        "total_sqft":           total_sqft,
        "living_to_lot_ratio":  living_to_lot_ratio,
        "basement_ratio":       basement_ratio,
        "city_encoded":         city_encoded,
    }

# ─────────────────────────────────────────────
#  LLM Interpretation
# ─────────────────────────────────────────────
async def get_llm_interpretation(h: HouseInput, features: dict, cluster_name: str, predicted_price: float) -> str:
    if not ANTHROPIC_API_KEY:
        return "⚠️ Set ANTHROPIC_API_KEY to enable AI interpretation."

    prompt = f"""You are a professional real estate analyst. Analyze this property and explain the predicted price.

Property Details:
- Living area: {h.living_area_m2:,.0f} m², Lot area: {h.lot_area_m2:,.0f} m²
- Bedrooms: {features['bedrooms']}, Bathrooms: {features['bathrooms']}
- Floors: {features['floors']}, Condition: {features['condition']}/5
- View score: {features['view']}/4, Waterfront: {'Yes' if features['waterfront'] else 'No'}
- House age: {features['house_age']} years, Renovated: {'Yes' if features['is_renovated'] else 'No'}
- Market segment: {cluster_name}

Predicted Price: ${predicted_price:,.0f}

In 3-4 sentences, explain:
1. Why this price makes sense for this property
2. Which 2-3 features had the most impact
3. Whether this seems realistic for the market segment

Keep it concise and professional."""

    async with httpx.AsyncClient() as client:
        resp = await client.post(
            "https://api.anthropic.com/v1/messages",
            headers={
                "x-api-key": ANTHROPIC_API_KEY,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": "claude-sonnet-4-20250514",
                "max_tokens": 300,
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=30,
        )
        data = resp.json()
        return data["content"][0]["text"]

# ─────────────────────────────────────────────
#  Endpoints
# ─────────────────────────────────────────────
@app.get("/")
def root():
    return {"message": "Real Estate ML API is running 🏠", "version": "1.0.0"}


@app.get("/cities")
def get_cities():
    """Returns list of all known cities."""
    return {"cities": sorted(le_city.classes_.tolist())}


@app.post("/predict", response_model=PredictResponse)
async def predict(house: HouseInput):
    try:
        # 1. Feature engineering
        feats = engineer_features(house)

        # ── Clustering ──────────────────────────────
        cluster_input = pd.DataFrame([feats])[scale_cols]
        cluster_input = cluster_input.fillna(cluster_input.median())
        X_scaled_cont = scaler.transform(cluster_input)

        binary_vals = np.array([[house.waterfront, feats["is_renovated"]]])
        X_scaled    = np.hstack([X_scaled_cont, binary_vals])

        cluster_id   = int(classifier.predict(X_scaled)[0])
        cluster_name = cluster_names[str(cluster_id)]

        # ── Regression ──────────────────────────────
        feats["cluster"] = cluster_id
        reg_df = pd.DataFrame([feats])[regression_features]
        reg_df = reg_df.fillna(train_medians)

        # Keep only selected features (after permutation importance)
        reg_cols_available = [c for c in selected_features if c in reg_df.columns]
        reg_df = reg_df.reindex(columns=selected_features, fill_value=0)

        log_price       = reg_model.predict(reg_df)[0]
        predicted_price = float(np.expm1(log_price))

        # ── LLM ─────────────────────────────────────
        interpretation = await get_llm_interpretation(house, feats, cluster_name, predicted_price)

        return PredictResponse(
            cluster_id      = cluster_id,
            cluster_name    = cluster_name,
            predicted_price = round(predicted_price, 2),
            price_formatted = f"${predicted_price:,.0f}",
            interpretation  = interpretation,
            # Return m² values so frontend always shows m², never sqft
            features_used   = {
                "living_area_m2":   house.living_area_m2,
                "lot_area_m2":      house.lot_area_m2,
                "above_area_m2":    house.above_area_m2,
                "basement_area_m2": house.basement_area_m2,
                "bedrooms":         feats["bedrooms"],
                "bathrooms":        feats["bathrooms"],
                "floors":           feats["floors"],
                "waterfront":       feats["waterfront"],
                "view":             feats["view"],
                "condition":        feats["condition"],
                "house_age":        feats["house_age"],
                "is_renovated":     feats["is_renovated"],
                "cluster_name":     cluster_name,
            },
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


@app.get("/cluster-info")
def cluster_info():
    """Returns info about each cluster for the frontend to display."""
    return {
        "clusters": {
            "0": {"name": "Upper Mid-range New",  "emoji": "🏡", "color": "#4ECDC4"},
            "1": {"name": "Luxury Spacious",       "emoji": "🏰", "color": "#FFD700"},
            "2": {"name": "Budget Old City",       "emoji": "🏘️",  "color": "#FF6B6B"},
            "3": {"name": "Mid-range Suburban",    "emoji": "🏠", "color": "#96CEB4"},
        }
    }